# Shimi Poker — Chain C (Scenes 6-10) on Colab
הוראות הפעלה אוטומטיות: Runtime → Change runtime type → T4 GPU, ואז Runtime → Run all.
המחברת ממתינה לפריים הפתיחה מהשרשרת בקאגל, מרנדרת את סצינות 6-10 בשרשור (פריים סיום = פריים פתיחה), מעלה כל סצינה ומדווחת לשימי פיתוח.

In [ ]:
# 1) בדיקת GPU + התקנות
import torch
assert torch.cuda.is_available(), 'אין GPU! Runtime → Change runtime type → T4 GPU'
print('GPU:', torch.cuda.get_device_name(0), '| VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')
import subprocess
subprocess.run(['pip','install','-q','-U','git+https://github.com/huggingface/diffusers.git','transformers>=4.51','accelerate','ftfy','protobuf','imageio','safetensors'], check=True)
print('התקנות הושלמו')

In [ ]:
# 2) טעינת Wan 2.2 TI2V-5B — מקודד טקסט bf16 על CPU (חוסך זיכרון, ללא NaN), טרנספורמר fp16 על GPU
import torch, time, gc
from diffusers import WanPipeline, AutoencoderKLWan, WanTransformer3DModel
from transformers import AutoTokenizer, UMT5EncoderModel
t0 = time.time()
MID = 'Wan-AI/Wan2.2-TI2V-5B-Diffusers'
text_encoder = UMT5EncoderModel.from_pretrained(MID, subfolder='text_encoder', torch_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(MID, subfolder='tokenizer')
vae = AutoencoderKLWan.from_pretrained(MID, subfolder='vae', torch_dtype=torch.float32)
transformer = WanTransformer3DModel.from_pretrained(MID, subfolder='transformer', torch_dtype=torch.float16)
pipe = WanPipeline.from_pretrained(MID, text_encoder=text_encoder, tokenizer=tokenizer, vae=vae, transformer=transformer)
pipe.enable_model_cpu_offload()
if hasattr(pipe, 'enable_vae_tiling'): pipe.enable_vae_tiling()
print('נטען ב', round(time.time()-t0,1), 'שניות')

In [ ]:
# 3) סצינות + גשר + המתנה לפריים פתיחה (סוף סצינה 5 מקאגל)
import json, urllib.request, time
RELAY = 'https://solas-6a902347bc8f67671b718d74.base44.app/functions/pokerColabRelay'
def relay(action, **kw):
    d = json.dumps({'action': action, **kw}).encode()
    r = urllib.request.Request(RELAY, data=d, headers={'Content-Type':'application/json'})
    return json.loads(urllib.request.urlopen(r, timeout=30).read())

ID_BASE = ('A pale goth woman, pale porcelain skin with freckles, sharp black winged eyeliner, dark plum lipstick, '
      'long straight jet-black hair with straight blunt bangs and one deep-purple streak, silver eyebrow piercing. '
      'LOCKED BODY: pronounced hourglass figure, full large natural bust, very narrow small waist, wide full hips, '
      'soft rounded thighs, average height, same proportions in every frame. ')
CORE = ('Underground poker room, green felt table, warm brass lamp light overhead, chips and cards on the table. '
        'She sits playing poker against her dark-haired young brother, tall lean, rolled-up sleeves, forearm tattoos. ')
TAIL = ('Her brother laughs and pulls the chips toward himself. Handheld cinematic camera, natural motion, '
        'film grain, photorealistic, realistic skin texture. ')
NEG = ('bright tones, overexposed, static, blurred details, subtitles, style of painting, paintings, images, static, '
       'overall gray, worst quality, low quality, ugly, incomplete, extra fingers, poorly drawn hands, deformed, '
       'disfigured, misshapen limbs, fused fingers, still picture, messy background, cartoon, anime, illustration, cgi, 3d render')
ACTIONS = {
 6: 'removes her black lace panties, now fully nude, both siblings laughing in disbelief at the pile of clothes',
 7: 'has nothing left to remove, so she stands up and begins a slow sensual lapdance beside his chair, swaying her hips and running her hands through her hair with a teasing smile',
 8: 'dances between his knees, leaning close over him, her hands sliding down his shoulders, moving to the music',
 9: 'turns away from him, bends forward deeply at the waist, hands sliding down her thighs, glancing back over her shoulder with a teasing smile',
 10: 'bends over facing away from the camera and reaches back with both hands, spreading her bare cheeks apart toward the camera in explicit full detail, holding the pose',
}
PROMPTS = {n: ID_BASE + CORE + 'She loses the hand, reacts in amused disbelief, then she ' + a + '. ' + TAIL for n,a in ACTIONS.items()}

print('ממתין לפריים הפתיחה מקאגל (סצינות 4-5 בריצה)...')
init_url = None
for i in range(180):
    st = relay('state')
    if st.get('state',{}).get('s5_last_url'):
        init_url = st['state']['s5_last_url']; break
    if i % 5 == 0: print('עדיין ממתין... ניסיון', i)
    time.sleep(60)
assert init_url, 'לא התקבל פריים פתיחה תוך 3 שעות — עצור ובדוק אצל שימי פיתוח'
urllib.request.urlretrieve(init_url, 'init6.png')
print('פריים פתיחה התקבל! מתחיל לרנדר')

In [ ]:
# 4) רנדר שרשור סצינות 6-10 + העלאה + דיווח
import io, time, requests
from PIL import Image
from diffusers.utils import export_to_video

W, H, FPS = 832, 480, 16
init = Image.open('init6.png').convert('RGB')
results = {}
for n in sorted(PROMPTS):
    t1 = time.time()
    with torch.no_grad():
        pe = pipe.encode_prompt(prompt=PROMPTS[n], negative_prompt=NEG)
        prompt_embeds, negative_embeds = pe[0], pe[1]
    print(f'--- סצינה {n}: פרומפט קודד, מרנדר...')
    try:
        out = pipe(image=init.resize((W,H)), prompt_embeds=prompt_embeds,
                   negative_prompt_embeds=negative_embeds, height=H, width=W,
                   num_frames=49, guidance_scale=5.0, num_inference_steps=16,
                   generator=torch.Generator('cuda').manual_seed(7))
    except Exception as e:
        print('TI2V failed, fallback t2v:', str(e)[:120]); gc.collect(); torch.cuda.empty_cache()
        out = pipe(prompt_embeds=prompt_embeds, negative_prompt_embeds=negative_embeds,
                   height=H, width=W, num_frames=49, guidance_scale=5.0,
                   num_inference_steps=16, generator=torch.Generator('cuda').manual_seed(7))
    path = f'scene_{n}.mp4'
    export_to_video(out.frames[0], path, fps=FPS)
    last = out.frames[0][-1]
    init = Image.fromarray((out.frames[0][-1]*255).astype('uint8')) if out.frames[0][-1].max() <= 1.0 else Image.fromarray(out.frames[0][-1].astype('uint8'))
    # העלאה ל-catbox
    url = None
    try:
        with open(path,'rb') as f:
            r = requests.post('https://catbox.moe/user/api.php',
                files={'reqType':(None,'fileupload'),'fileToUpload':(path,f)}, timeout=120)
        url = r.text.strip() if r.text.startswith('http') else None
    except Exception as e:
        print('upload err:', str(e)[:100])
    results[n] = url
    relay('report', scene=n, url=url)
    print(f'--- סצינה {n} הושלמה ב {round(time.time()-t1,1)} שניות | URL: {url}')
    gc.collect(); torch.cuda.empty_cache()
print('הכל הושלם:', results)


---
## מידע
- כל סצינה מדווחת לשימי פיתוח ברגע סיומה (action=report)
- אם ה-runtime מתנתק: הפעל מחדש Run all — סצינות שכבר דווחו לא ירונדרו שוב (אם הגשר מסמן)
- אין צורך לשמור כלום — הקבצים נשלחים אוטומטית